In [ ]:
#import dependencies, set paths
import os
from os.path import join
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import ImageGrid, make_axes_locatable
import pickle
import mne
from mne import read_evokeds, SourceEstimate
from mne.stats import summarize_clusters_stc

In [ ]:
## modify the test variables here
hemi = 'lh'
use_roi = True # subset stcs with spatial_exclude in test
search_tmin = 0
search_tmax = 400

In [ ]:
expt = 'EXPT'
ROOT = f'/path/to/{expt}'
os.chdir(ROOT)

subjects_dir = join(ROOT,'mri')
log_dir  = join(ROOT, 'logs')
stc_dir = join(ROOT, 'stc')
stats_dir = join(ROOT, 'stats')

excluded = ['R0000']
subjects = [i[:5] for i in os.listdir(subjects_dir) if i.startswith('R') and i[:5] not in excluded and not i.endswith(".fif")]

output_dir = os.path.join(ROOT, 'figures')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# permutation test vars 
tail = 1
p_thresh = 0.05
n_permutations = 10000


In [ ]:
def load_source_space(subjects_dir):
    """Load source space information."""
    src_fname = os.path.join(subjects_dir, 'fsaverage', 'bem', 'fsaverage-ico-4-src.fif')
    return mne.read_source_spaces(src_fname)

src = load_source_space(subjects_dir)
fsave_vertices = [s['vertno'] for s in src]
lh_vertices = fsave_vertices[0]
rh_vertices = fsave_vertices[1]

In [ ]:
def get_STC(subj, cond):
    stc_fname = os.path.join(stc_dir, '%s', '%s_%s_dSPM') % (cond, subj, cond)
    stc = mne.read_source_estimate(stc_fname, subject='fsaverage')
    return stc

In [ ]:
# brodmann areas of interest
brodmann_areas = ['17', '20', '21', '22', '38', '39', '11', '44', '45']

# define colors for conditions
colors = {
    'CONDITION1': 'tab:blue',
    'CONDITION2': 'tab:orange',
    'CONDITION3': 'tab:green',
    'CONDITION4': 'tab:red'
}

conditions = list(colors.keys())

In [ ]:
pickle_fnames = [i for i in os.listdir(stats_dir) if i.endswith(".pickled")]
for num in brodmann_areas:
    annot_name = 'PALS_B12_Brodmann'
    labels = mne.read_labels_from_annot('fsaverage', annot_name, subjects_dir=subjects_dir, hemi=hemi)
    label_name = f'Brodmann.{num}-{hemi}'
    roi = [label for label in labels if label.name==label_name][0]

    # vertices in roi
    n_verts = 2562
    idx = np.arange(0, n_verts, 1)
    roi_idx = roi.get_vertices_used(vertices=idx)
    diff_idx = np.setdiff1d(idx, roi_idx)

    for pkl_fname in pickle_fnames:
        if num in pkl_fname:
            print(pkl_fname)
            hemi = 'lh' if 'lh' in pkl_fname else 'rh'

            # load clusters
            with open(join(stats_dir, pkl_fname), "rb") as file:
                F_obs, clusters, pvals, h0 = pickle.load(file)

            tstep = 0.01
            p_thresh = 0.2 # generous threshold
            print(pvals)

            sig_clusters = np.where(pvals < p_thresh)[0]
            np.set_printoptions(suppress=True)
            print(sig_clusters)

            if len(sig_clusters) > 0:
                for ci in sig_clusters:
                    cluster = clusters[ci]
                    pval = pvals[ci]
                    F_cluster = F_obs
                    h0_cluster = h0

                    t_start = cluster[0][0]
                    t_end = cluster[0][-1]

                    print(f"Cluster {ci}: {t_start} - {t_end} ms, p-value: {pval}")

                    # ensure 2D
                    if len(F_cluster.shape) == 1:
                        F_cluster = F_cluster[:, np.newaxis]

                    hv = fsave_vertices[1:] if hemi == 'lh' else fsave_vertices[:1]

                    # cluster data
                    c_data = (F_cluster, [cluster], np.array([pval]), h0_cluster)
                    stc_vis = summarize_clusters_stc(c_data, p_thresh=p_thresh, tstep=tstep, vertices=hv, subject="fsaverage")
                    c_data = stc_vis.data

                    lv = fsave_vertices[0]
                    rv = fsave_vertices[1]

                    # zero data for other hemisphere
                    zeros = np.zeros((len(lv), 2))
                    c_data_both = np.vstack([c_data, zeros]) if hemi == 'lh' else np.vstack([zeros, c_data])

                    # source estimate
                    stc = SourceEstimate(c_data_both, [lv, rv],
                                        tmin=stc_vis.tmin,
                                        tstep=stc_vis.tstep,
                                        subject=stc_vis.subject)

                    # plot
                    brain = stc.plot(hemi='both', subjects_dir=subjects_dir,
                                    time_viewer=True, clim='auto')
                    brain.show()

                    # get anatomical labels
                    labels_lh, labels_rh = mne.stc_to_label(
                        stc,
                        src=src,
                        smooth=True,
                        subjects_dir=subjects_dir,
                        connected=True,
                        verbose="error",
                    )

                    # select hemisphere labels
                    func_labels = labels_lh if hemi == "lh" else labels_rh
                    func_label = func_labels[0]
                    fname = f"stats_clu{ci}_p{pval}-{hemi}.label"
                    print(fname)
                    save = input("save label? ")

                    if save == 'y':
                        func_label.save(subjects_dir, 'fsaverage', 'label', fname)
                    else:
                        print("not saving...")

                    # ensure 2D
                    if len(F_obs.shape) == 1:
                        F_obs = F_obs[:, np.newaxis]

                    # extract cluster
                    clu_data = (F_obs, [cluster], np.array([pval]), h0)
                    v = fsave_vertices[1:] if hemi == 'rh' else fsave_vertices[:1]
                    stc_vis = summarize_clusters_stc(clu_data, p_thresh=p_thresh, tstep=0.01,
                                                   vertices=v, subject="fsaverage")
                    
                    c_data = stc_vis.data
                    lv = stc_vis.vertices[0]
                    rv = stc_vis.vertices[0]
                    
                    # hemisphere data
                    zeros = np.zeros((len(rv), 2))
                    lh_data, rh_data = (c_data, zeros) if hemi == 'lh' else (zeros, c_data)
                    all_data = np.vstack([lh_data, rh_data])

                    # create visualization stc
                    clu_stc = SourceEstimate(all_data, [lv, rv],
                                           tmin=stc_vis.tmin,
                                           tstep=stc_vis.tstep,
                                           subject=stc_vis.subject)
                    
                    # get color limits for legend
                    tmp_brain = clu_stc.plot(hemi=hemi, subjects_dir=subjects_dir,
                                           time_viewer=False, clim='auto',
                                           background='white', colorbar=False, views='lateral')
                    
                    fmin, fmid, fmax = tmp_brain._data['fmin'].round(2), tmp_brain._data['fmid'].round(2), tmp_brain._data['fmax'].round(2)
                    cmap = tmp_brain._data['colormap']
                    tmp_brain.close()

                    clim = dict(kind='value', lims=(fmin, fmid, fmax))

                    # brain views
                    img_list = []
                    for view in ['lateral', 'ventral']:
                        brain = clu_stc.plot(
                            hemi=hemi,
                            subjects_dir=subjects_dir,
                            time_viewer=False,
                            clim='auto',
                            background='white',
                            colorbar=False,
                            views=view
                        )
                        mask = mne.Label(hemi=hemi, vertices=diff_idx, subject='fsaverage').fill(src=src).smooth()
                        if hemi == 'lh':
                            brain.add_label(mask, borders=False, color='black', alpha=0.8)
                        
                        shot = brain.screenshot()
                        brain.close()

                        # crop image
                        rm_pix = (shot != 255).any(-1)
                        rm_row = rm_pix.any(1)
                        rm_col = rm_pix.any(0)
                        img = shot[rm_row][:, rm_col]

                        img_list.append(np.rot90(img) if view == 'ventral' else img)

                    # figure layout
                    fig = plt.figure(figsize=(12, 4))
                    gs = GridSpec(3, 3, width_ratios=[12, 3, 3], height_ratios=[1, 0.5, 0.1], wspace=0.2) 
                    
                    # time arrays
                    times = np.linspace(-100, 800, 901)
                    t = len(times)

                    # extract time series
                    subj_data = []
                    for s in subjects:
                        cond_data = []
                        for i, cond in enumerate(conditions):
                            fname = ROOT + f'data/stc/{s}/{cond}/{s}_{cond}_dSPM'
                            stc = mne.read_source_estimate(fname)
                            
                            # get timeseries data
                            disp_t = stc.times * 1000  # ms
                            ts = stc.extract_label_time_course(labels=func_label, src=src, mode="auto")
                            
                            cond_data.append(ts)
                        subj_data.append(cond_data)
                        
                    # average data
                    subj_data = np.array(subj_data)
                    avg = subj_data.mean(axis=0).reshape((len(conditions), t))
                    avg = np.squeeze(avg)
                    subj_data = subj_data.reshape((len(subjects), len(conditions), t))
                    
                    # display window
                    tmin_ep = -100
                    tmax_ep = 800
                    
                    print(f"Analysis window: {search_tmin} to {search_tmax} ms")
                    
                    # search indices
                    adj_tmin = times[0] + search_tmin
                    adj_tmax = times[0] + search_tmax
                    
                    # indices
                    tmin_i = np.where(disp_t >= adj_tmin)[0][0]
                    tmax_i = np.where(disp_t <= adj_tmax)[0][-1]
                    
                    ep_tmin_i = np.where(disp_t >= tmin_ep)[0][0]
                    ep_tmax_i = np.where(disp_t <= tmax_ep)[0][-1]

                    # waveform plot
                    ax_wave = fig.add_subplot(gs[:, 0])
                    for i, cond in enumerate(conditions):
                        ax_wave.plot(
                            disp_t[ep_tmin_i:ep_tmax_i + 1], 
                            avg[i, ep_tmin_i:ep_tmax_i + 1],
                            label=cond, 
                            color=colors[cond]
                        )

                    # cluster highlight
                    ax_wave.axvspan(t_end, t_start, color='gray', alpha=0.25)
                    ax_wave.set_ylabel("dSPM")
                    ax_wave.set_xlabel("Time (ms)")
                    ax_wave.set_title('')
                    ax_wave.legend(fontsize=10)
                    ax_wave.set_xlim(tmin_ep, tmax_ep)

                    # bar plot
                    ax_bar = fig.add_subplot(gs[:, 1])
                    means = [avg[i, tmin_i:tmax_i].mean() for i in range(len(conditions))]
                    stds = avg.std(axis=1) / np.sqrt(len(subjects))
                    ax_bar.bar(range(len(conditions)), means, yerr=stds, color=[colors[cond] for cond in conditions], alpha=0.9, capsize=5)
                    ax_bar.set_xticks(range(len(conditions)))
                    ax_bar.set_xticklabels("")
                    ax_bar.set_ylabel("dSPM")
                    ax_bar.set_title('')
                    
                    # brain views
                    ax_lat = fig.add_subplot(gs[0, 2])
                    ax_vent = fig.add_subplot(gs[1, 2])

                    for ax, img in zip([ax_lat, ax_vent], img_list):
                        ax.imshow(img)
                        ax.axis('off')
                    
                    # colorbar
                    ax_cbar = fig.add_subplot(gs[2, 2])
                    ax_cbar.axis("off")
                    cax = make_axes_locatable(ax_cbar).append_axes("bottom", size="100%", pad=0)
                    mne.viz.plot_brain_colorbar(cax, clim=clim, colormap=cmap, label="Activation (F)", orientation="horizontal")
                    ax.set_xlabel("Time (ms)")
                    ax.set_ylabel("dSPM")

                    # title
                    fig.suptitle(f"{label_name}, {t_start}-{t_end} ms, p = {pval}")
                    plt.close('all')
        else:
            continue